In [ ]:
# Locate the repository when this historical notebook is opened in its folder.
from pathlib import Path
import os, sys
_repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'cps_paper_algorithms.py').is_file())
os.chdir(_repo)
sys.path[:0] = [str(_repo / 'src'), str(_repo)]


# Protein backbones: global GCS, CPS-3F and thesis CPS-2F

This notebook compares three **different** optimization problems on decimated, ordered 3D Cα curves. Coordinates, arc lengths and distance thresholds are in **ångströms (Å)**. It runs every unique pair from Fan et al. §7, Tables 1–3 (14 pairs; Tables 1 and 2 reuse seven pairs).

The executable default is `w=8`, a **coarse-trace/runtime experiment**, with `α ∈ {1,2}`. This is not a biological headline result. Change `WS` below to run other supplied levels, including the `w=1` control. Strides 8 and 16 must remain labelled runtime-only. No additional decimation is applied to the supplied CSV curves.

**Inputs:** by default, use every vertex of the supplied **aligned** decimated CSV curves, including their original endpoints and any gap-crossing edges. Tables 1–2 supply seven pairs. The seven Table 3 pairs are supplemented from the full cached audited chains, decimated separately with endpoints retained; unequal lengths are supported. Their cached rigid US-align transforms are held fixed. No historical result counts are treated as new measurements. Set `INPUT_SCOPE="qualified"` to use the previous continuous-interval preparation instead.

Sources: [van de Kerkhof et al., Global Curve Simplification, Algorithm 1](../../docs/RESEARCH_SOURCES.md#global-curve-simplification); [Fan et al., full version, §4 Algorithm 1 and §7](https://arxiv.org/pdf/1409.2457); [Galit Gozoltzani, local thesis, Chapters 3–5, Algorithms 1–2](../../docs/RESEARCH_SOURCES.md#thesis). The thesis file is an annotated draft with unresolved editorial/proof comments. Its configuration graph is implemented as specified; small-instance tests support implementation correctness, not a new proof of its discretization theorem.

**Executed configuration:** w=8, α=1 and 2; results are written to `output/cps_papers_w8/`. Earlier w=16 artifacts remain in `output/cps_papers/`.

## 1. Load data and configure the experiment

Use the project environment in `requirements-analysis.txt`. The default loads the supplied aligned CSV coordinates without cropping or another decimation. Table 3 supplements use full cached residue coordinates and a previously computed rigid alignment. The coordinate-frame provenance is recorded per pair. The curves are mathematical polylines: edges crossing missing residues remain edges in this default experiment.

In [ ]:
from pathlib import Path
import json, hashlib, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    raise RuntimeError('Run this notebook from the project root.')
OUT = ROOT / 'output' / 'cps_papers_w8'
OUT.mkdir(parents=True, exist_ok=True)
WS = (8,)  # Run the supplied stride-8 inputs
ALPHAS = (1, 2)
INCLUDE_TABLE3 = True
INPUT_SCOPE = "supplied"  # or "qualified": previously audited common continuous intervals
BUDGETS = dict(max_states=300_000, max_transitions=15_000_000, seconds=90.)
from cps_notebook_data import load_inputs
data = load_inputs(ROOT, WS, INCLUDE_TABLE3, scope=INPUT_SCOPE)

## 2. General properties — three cells

Printed names are preserved beside the verified PDB/author-chain interpretation. A PDB-cache SHA-256 check prevents silently reusing an audit against changed structures. Current deposited lengths can differ from reported lengths; none is trimmed merely to match the paper.

In [ ]:
display(pd.DataFrame({
    'property': ['CSV vertices', 'CSV chain/frame/stride groups', 'CSV pair/frame/stride groups',
                 'unique selected pairs', 'executed strides', 'coordinate units'],
    'value': [len(data['curves']), len(data['chain_summary']), len(data['supplied_pairs']),
              data['inputs'].pair.nunique(), str(WS), 'Å']}))
display(data['inputs'][['B','tables','w','nA','nB','role']])

In [ ]:
display(data['audit'][['printed','verified','reported_length','deposited_ca_length',
    'prepared_ca_length','length_matches_paper','author_chain','label_chain','model','fragments','interpretation']])

In [ ]:
display(data['inputs'][['B','origin','dropped_decimated_vertices_per_chain',
    'source_first','source_last','full_chain_endpoints_preserved','fragments_A','fragments_B']])
display(data['paper'])  # historical thresholds and results, NOT newly computed results

## 3. Organize the curves for consumption

`instances["1o7j.a__1hfj.c@8"]` holds `A`, `B_xyz` and the two residue maps. `input_index` is the solver index; `curve_index` is the original source-row index. Author residue number, insertion code, label residue number, model and chosen alternate atom are retained.

The residue maps preserve first-model Cα selections and document missing residues, alternate atom choices and fragment IDs. In `supplied` mode, gaps are reported but not removed; the exact input polyline is simplified. Source row 0 and the final source row are retained. Table 3 curves are sampled independently, so A and B can have different lengths; discrete Fréchet coupling does not require an index-wise correspondence. In `qualified` mode, the earlier common continuous intervals and their exclusions are used instead.

In [ ]:
instances = data['instances']
example = next(iter(instances.values()))
display(data['residues'][['instance','side','input_index','curve_index','pdb_id','model',
    'author_chain','label_chain','author_seq_id','insertion_code','label_seq_id','altloc','fragment_id']].head(10))
print('Example array shapes:', example['A'].shape, example['B_xyz'].shape)
display(Markdown('Detailed source records: [chain breaks](output/decimation/chain_breaks.csv), '
 '[missing residues](output/decimation/missing_residues.csv), '
 '[qualification exclusions](output/decimation/qualification_excluded_residues.csv).'))

## 4. Algorithms and the comparison contract

| Method | Fidelity of A and B to their simplifications | Between the two outputs | Objective |
|---|---|---|---|
| Independent GCS | global continuous Fréchet | no joint constraint | minimize each vertex count separately |
| CPS-3F | global **discrete** Fréchet | discrete Fréchet ≤ δ₃ | minimize max(kA,kB) |
| Thesis CPS-2F | global **continuous** Fréchet | discrete Fréchet ≤ δ₃ | minimize max(kA,kB) |

All outputs are endpoint-preserving vertex subsequences of the prepared input. This is the anchored version used in the algorithms; the optional unanchored variants in the source remarks are not run. A discrete coupling may repeat an A or B index and permits `kA != kB`. Its maximum link is reported along the actual optimal coupling, not by zipping unequal arrays. `indexwise_max_link` is an additional diagnostic only when the sizes agree.

For each actual prepared `(pair,w)`, `e = (mean_edge(A)+mean_edge(B))/2`, `δ₁=δ₂=αe`, and `δ₃=ceil(d_dF(A,B))`. These are bond-scaled comparisons, not a numerical reproduction of the historical tables. The endpoint maximum is a necessary lower bound; a solver may still be infeasible because of fidelity constraints. With the current δ₃ construction the full input curves themselves supply a feasible discrete coupling.

**GCS:** the implementation realizes Algorithm 1's free-space cost envelope with exact integer link-count layers. Shortcut endpoints are input vertices, but their matched positions along the original curve are free to slide globally. A local shortcut path supplies only a feasible upper bound on the search depth. Time is O(n³K), memory O(n²K), K ≤ n−1; it is not a local shortcut-DAG solution.

**CPS-3F:** the explicit configuration graph from Fan et al. §4 Algorithm 1 has owner indices advancing by 0 or 1 and retained-vertex indices advancing by any nonnegative amount. It uses the paper's two-cost DP recurrence. This notebook does **not** claim the faster §5 Algorithm 2 implementation or its runtime bound.

**CPS-2F:** insert each original-vertex sphere's intersections with original edges as owner-only auxiliary points. These do not alter geometry and cannot be selected in the output. Construct the Chapter 5 graph using monotone transitions and continuous subcurve/segment tests. The dynamic program stores the minimum B-hop count at each exact A-hop count; the equivalent exact-count convention avoids conflating an 'at most' budget with initialization. Exclude self-loops, retain the source initialization, and add one to hop counts to report vertices.

For both joint methods we first compute independent minima under the matching fidelity measure. If that pair obeys δ₃, it attains a lower bound on the joint objective and is certified optimal immediately. For CPS-2F a bounded search may also find a different feasible pair at that same lower bound; this is labelled `optimal_alternate_certificate`, not graph execution. Otherwise the explicit configuration graph runs. `resource_limit` is never called infeasible and never assigned fabricated output metrics. The graph solver optimizes vertex count, not the smallest achievable pairing distance.

### 4.1 Global continuous Fréchet: free-space geometry and GCS

These executable definitions implement the global cost-layer sweep. Auxiliary functions include a continuous decision predicate and a discrete distance evaluator.

In [ ]:
"""Continuous global vertex-restricted simplification and discrete CPS references.

The GCS solver represents Algorithm 1's cost lower envelope by its integer
cost layers: reach[v,k,i] is the earliest reachable point on original edge i
at spine v using exactly k shortcut links. No shortcut/subcurve anchoring.
"""
import itertools
import numpy as np
from numba import njit

TOL = 1e-10

@njit(cache=False)
def ball_segment_interval(a,b,c,delta):
    """Closed parameter interval where segment a+t(b-a) is within delta of c."""
    d=b-a; q=a-c
    aa=np.dot(d,d)
    if aa<1e-24:
        if np.dot(q,q)<=delta*delta+TOL: return 0.,1.
        return np.inf,-np.inf
    center=-np.dot(q,d)/aa
    closest=q+center*d
    slack=delta*delta-np.dot(closest,closest)
    if slack < -TOL: return np.inf,-np.inf
    radius=np.sqrt(max(0.,slack)/aa)
    lo=max(0.,center-radius); hi=min(1.,center+radius)
    if lo>hi+TOL: return np.inf,-np.inf
    return min(lo,hi),hi

@njit(cache=False)
def segment_frechet_decision(P,a,b,delta):
    """Exact continuous decision for a polygonal curve versus one segment."""
    if np.linalg.norm(P[0]-a)>delta+TOL or np.linalg.norm(P[-1]-b)>delta+TOL: return False
    previous=0.
    for p in P:
        lo,hi=ball_segment_interval(a,b,p,delta)
        previous=max(previous,lo)
        if previous>hi+TOL: return False
    return True

@njit(cache=False)
def continuous_decision(P,Q,delta):
    """Full continuous free-space reachability, including degenerate edges."""
    n=len(P); m=len(Q)
    if n==1: return np.max(np.sqrt(np.sum((Q-P[0])**2,axis=1)))<=delta+TOL
    if m==1: return np.max(np.sqrt(np.sum((P-Q[0])**2,axis=1)))<=delta+TOL
    if np.linalg.norm(P[0]-Q[0])>delta+TOL or np.linalg.norm(P[-1]-Q[-1])>delta+TOL: return False
    right=np.full((n-1,m-1),np.inf)
    top=np.full((n-1,m-1),np.inf)
    for i in range(n-1):
        for j in range(m-1):
            if i==0:
                lo,hi=ball_segment_interval(Q[j],Q[j+1],P[0],delta)
                left=0. if lo<=TOL and (j==0 or top[0,j-1]<=TOL) else np.inf
            else: left=right[i-1,j]
            if j==0:
                lo,hi=ball_segment_interval(P[i],P[i+1],Q[0],delta)
                bottom=0. if lo<=TOL and (i==0 or right[i-1,0]<=TOL) else np.inf
            else: bottom=top[i,j-1]
            rlo,rhi=ball_segment_interval(Q[j],Q[j+1],P[i+1],delta)
            tlo,thi=ball_segment_interval(P[i],P[i+1],Q[j+1],delta)
            r=rlo if np.isfinite(bottom) else max(rlo,left)
            t=tlo if np.isfinite(left) else max(tlo,bottom)
            if r<=rhi+TOL: right[i,j]=r
            if t<=thi+TOL: top[i,j]=t
    return right[-1,-1]<=1.+TOL or top[-1,-1]<=1.+TOL

def continuous_distance(P,Q,absolute_tolerance=1e-5):
    """Certified decision bracket [lower, upper], not an exact critical value."""
    P=np.asarray(P,float); Q=np.asarray(Q,float)
    lower=max(np.linalg.norm(P[0]-Q[0]),np.linalg.norm(P[-1]-Q[-1]))
    upper=discrete_frechet(P,Q)
    if continuous_decision(P,Q,lower): return float(lower),float(lower)
    while upper-lower>absolute_tolerance:
        mid=(lower+upper)/2
        if continuous_decision(P,Q,mid): upper=mid
        else: lower=mid
    return float(lower),float(upper)

@njit(cache=False)
def local_shortcut_path(P,delta):
    """Restricted local baseline; used ONLY as a feasible link-count upper bound."""
    n=len(P); cost=np.full(n,n+1,np.int32); pred=np.full(n,-1,np.int32); cost[0]=0
    for v in range(1,n):
        for u in range(v):
            if cost[u]+1<cost[v] and segment_frechet_decision(P[u:v+1],P[u],P[v],delta):
                cost[v]=cost[u]+1; pred[v]=u
    path=[n-1]
    while path[-1]>0: path.append(pred[path[-1]])
    return np.array(path[::-1])

@njit(cache=False)
def _gcs(P,delta,cap):
    n=len(P); rows=n-1
    spine_lo=np.empty((n,rows)); spine_hi=np.empty((n,rows))
    for v in range(n):
        for i in range(rows):
            spine_lo[v,i],spine_hi[v,i]=ball_segment_interval(P[i],P[i+1],P[v],delta)
    reach=np.full((n,cap+1,rows),np.inf)
    parent_u=np.full((n,cap+1,rows),-1,np.int32)
    parent_row=np.full((n,cap+1,rows),-1,np.int32)
    # Starting spine: move along P while the simplification stays at P[0].
    for i in range(rows):
        if spine_lo[0,i]>TOL: break
        reach[0,0,i]=0.
        if spine_hi[0,i]<1.-TOL: break
    for v in range(1,n):
        for u in range(v):
            top_lo=np.empty(rows); top_hi=np.empty(rows)
            for i in range(rows):
                top_lo[i],top_hi[i]=ball_segment_interval(P[u],P[v],P[i+1],delta)
            for k in range(1,min(cap,v)+1):
                # Each strip cell is convex. Bottom arrival frees all of its right
                # boundary; left arrival constrains its right boundary from below.
                bottom=np.inf; origin=-1
                for i in range(rows):
                    left=reach[u,k-1,i]
                    if np.isfinite(bottom):
                        r=spine_lo[v,i]; source=origin
                    else:
                        r=max(spine_lo[v,i],left); source=i
                    if r<=spine_hi[v,i]+TOL and r<reach[v,k,i]:
                        reach[v,k,i]=r
                        parent_u[v,k,i]=u; parent_row[v,k,i]=source
                    if np.isfinite(left):
                        t=top_lo[i]; origin=i
                    else: t=max(top_lo[i],bottom)
                    bottom=t if t<=top_hi[i]+TOL else np.inf
        # A terminal point may lie at the top of the last original edge.
    best=cap
    for k in range(1,cap+1):
        if reach[n-1,k,rows-1]<=1.+TOL:
            best=k; break
    if not np.isfinite(reach[n-1,best,rows-1]): raise ValueError('No reachable terminal state')
    path=np.empty(best+1,np.int64); path[best]=n-1
    v=n-1; i=rows-1
    for k in range(best,0,-1):
        u=parent_u[v,k,i]; r=parent_row[v,k,i]
        path[k-1]=u; v=u; i=r
    return path

def shortest_independent(P,delta):
    """Shortest endpoint-preserving vertex subsequence under global continuous F.

    Algorithm 1 / Lemma 4 of the supplied full GCS paper, using integer cost
    layers instead of explicitly subdividing all elementary intervals. The layers
    here represent EXACT link counts (the minimum over them is the lower envelope).
    Time O(n^3 K), memory O(n^2 K), K <= n-1 is a feasible local upper bound.
    Floating-point geometric predicates have TOL=1e-10; no resampling is used.
    """
    P=np.ascontiguousarray(P,dtype=float)
    if delta<0 or not np.isfinite(delta) or len(P)==0 or not np.isfinite(P).all(): raise ValueError('Invalid curve/tolerance')
    if len(P)==1: return np.array([0],dtype=int)
    local=local_shortcut_path(P,delta)
    result=_gcs(P,float(delta),len(local)-1)
    if not continuous_decision(P,P[result],delta): raise AssertionError('Independent continuous validation failed')
    return result

@njit(cache=False)
def discrete_frechet(P,Q):
    prev=np.full(len(Q),np.inf)
    for i in range(len(P)):
        current=np.full(len(Q),np.inf)
        for j in range(len(Q)):
            if i==0 and j==0: preceding=0.
            else:
                preceding=prev[j]
                if j>0: preceding=min(preceding,current[j-1],prev[j-1])
            current[j]=max(preceding,np.linalg.norm(P[i]-Q[j]))
        prev=current
    return prev[-1]



### 4.2 Joint solvers: discrete and continuous configuration graphs

The two graph builders share the two-count dynamic program. `certificate=False` forces graph execution; the default also accepts a feasible pair attaining the independent lower bound.

In [ ]:
"""Paper configuration DAGs; continuous GCS is in curve_algorithms.py.

CPS-3F: Fan et al. section 4, Algorithm 1 (explicit graph version).
CPS-2F: Gozoltzani local thesis chapters 3--5, Algorithms 1--2.
Anchored endpoints; asynchronous discrete coupling; objective max(kA,kB).
"""
import heapq
import itertools
import time
import numpy as np
from numba import njit
# Geometry functions are defined in the preceding cell.

EPS = 1e-9

class ResourceLimit(RuntimeError):
    pass

def extended_curve(P, delta):
    """Original vertices plus exact sphere/edge intersections, ordered on P.

    Auxiliary points are owner positions ONLY, never output candidates.
    Store curve parameters to disambiguate coincident spatial points.
    """
    parameters = list(map(float, range(len(P))))
    for i in range(len(P)-1):
        a,b=P[i:i+2]; v=b-a; aa=float(v@v)
        if aa < 1e-24:
            continue  # zero edge has identical geometry; preserve its original ends
        for c in P:
            q=a-c; center=-float(q@v)/aa
            slack=delta*delta-float((q+center*v)@(q+center*v))
            if slack < -1e-10: continue
            radius=np.sqrt(max(0.,slack)/aa)
            for t in (center-radius,center+radius):
                if EPS<t<1-EPS: parameters.append(i+t)
    ordered=[]
    for t in sorted(parameters):
        if not ordered or t-ordered[-1]>1e-10: ordered.append(t)
    t=np.asarray(ordered); i=np.minimum(t.astype(int),len(P)-2)
    X=P[i]+(t-i)[:,None]*(P[i+1]-P[i])
    return X,t

@njit(cache=False)
def _discrete_edges(P,d,max_edges):
    n=len(P); edges=[]
    for i in range(n):
        for p in range(n):
            if np.linalg.norm(P[i]-P[p])>d+EPS: continue
            u=i*n+p
            for ii in range(i,min(n,i+2)):
                for pp in range(p,n):
                    if np.linalg.norm(P[ii]-P[pp])<=d+EPS:
                        edges.append((u,ii*n+pp,int(pp>p)))
                        if len(edges)>max_edges:return edges,False
    return edges,True

@njit(cache=False)
def _continuous_edges(P,X,d,max_edges):
    """Incremental exact curve/segment test; includes zero-cost owner motion."""
    n=len(P); h=len(X); edges=[]
    for p in range(n):
        for pp in range(p,n):
            lo=np.empty(h); hi=np.empty(h)
            for ii in range(h):
                lo[ii],hi[ii]=ball_segment_interval(P[p],P[pp],X[ii],d)
            for i in range(h):
                if np.linalg.norm(X[i]-P[p])>d+EPS: continue
                previous=0.
                for ii in range(i,h):
                    previous=max(previous,lo[ii])
                    if previous>hi[ii]+1e-10: break
                    if np.linalg.norm(X[ii]-P[pp])<=d+EPS:
                        edges.append((i*n+p,ii*n+pp,int(pp>p)))
                        if len(edges)>max_edges: return edges,False
    return edges,True

def chain_graph(P,d,kind,max_edges=2000000):
    P=np.ascontiguousarray(P,float)
    if kind=='CPS-2F':
        X,t=extended_curve(P,d)
        edges,finished=_continuous_edges(P,X,float(d),max_edges)
        if not finished: raise ResourceLimit('single-chain graph edge budget exceeded')
    elif kind=='CPS-3F':
        X=P; t=np.arange(len(P),dtype=float); edges,finished=_discrete_edges(P,float(d),max_edges)
        if not finished:raise ResourceLimit('single-chain graph edge budget exceeded')
    else: raise ValueError(kind)
    size=len(X)*len(P); out=[[] for _ in range(size)]; incoming=[[] for _ in range(size)]
    for u,v,c in edges:
        u,v,c=int(u),int(v),int(c)
        if u!=v: out[u].append((v,c));incoming[v].append(u)
    # Only states on an anchored source-to-terminal path can matter.
    forward=np.zeros(size,bool);forward[0]=True
    for u in range(size):
        if forward[u]:
            for v,c in out[u]: forward[v]=True
    backward=np.zeros(size,bool);backward[-1]=True
    for v in range(size-1,-1,-1):
        if backward[v]:
            for u in incoming[v]:backward[u]=True
    active=forward&backward
    for u in range(size):
        out[u]=([(u,0)]+[(v,c) for v,c in out[u] if active[v]]) if active[u] else []
    return dict(P=P,X=X,parameters=t,out=out,size=size,active=active,
                edges=sum(map(len,out)),auxiliary=len(X)-len(P))

def independent_discrete(P,d):
    """Exact shortest anchored discrete simplification, single configuration DAG."""
    g=chain_graph(P,d,'CPS-3F'); n=len(P)
    cost=np.full(g['size'],n+1,int); pred=np.full(g['size'],-1,int);cost[0]=0
    for u in range(g['size']):
        for v,c in g['out'][u]:
            if u!=v and cost[u]+c<cost[v]:cost[v]=cost[u]+c;pred[v]=u
    u=g['size']-1; path=[]
    while u>=0:path.append(u%n);u=pred[u]
    return np.asarray(list(dict.fromkeys(path[::-1])),int)

def discrete_coupling(A,B):
    n,m=len(A),len(B);D=np.full((n,m),np.inf);pred={}
    for i in range(n):
        for j in range(m):
            if i==j==0:D[i,j]=np.linalg.norm(A[i]-B[j]);continue
            candidates=[(D[u,v],u,v) for u,v in ((i-1,j),(i,j-1),(i-1,j-1)) if u>=0 and v>=0]
            value,u,v=min(candidates);D[i,j]=max(value,np.linalg.norm(A[i]-B[j]));pred[i,j]=(u,v)
    path=[(n-1,m-1)]
    while path[-1]!=(0,0):path.append(pred[path[-1]])
    return np.asarray(path[::-1],int),float(D[-1,-1])

def alternate_certificate(A,B,d1,d2,d3,ia,ib,limit=150000):
    """Optional exact lower-bound witness search; does not claim graph execution.

    Any feasible pair with max count equal to the independent lower bound is
    globally optimal. Failure to find one proves nothing and falls back to DP.
    """
    bound=max(len(ia),len(ib));tested=0;sets=[]
    for P,Q,d,base,other in [(A,B,d1,ia,ib),(B,A,d2,ib,ia)]:
        choices=[base]
        for k in range(len(base),bound+1):
            for middle in itertools.combinations(range(1,len(P)-1),k-2):
                ids=np.array((0,)+middle+(len(P)-1,));tested+=1
                if tested>limit:return None,tested
                if not continuous_decision(P,P[ids],d):continue
                if discrete_frechet(P[ids],Q[other])<=d3+EPS:
                    return ((ids,other) if len(sets)==0 else (other,ids)),tested
                choices.append(ids)
        sets.append(choices)
    for a in sets[0]:
        for b in sets[1]:
            tested+=1
            if tested>limit:return None,tested
            if discrete_frechet(A[a],B[b])<=d3+EPS:return (a,b),tested
    return None,tested

def solve_cps(A,B,d1,d2,d3,kind,*,certificate=True,max_states=300000,
              max_transitions=15000000,seconds=90.,max_chain_edges=2000000):
    """Same min/max-count recurrence as paper DP, sparse reachable states.

    X[state][r] stores minimum B hops at EXACT A hop count r. This is the
    equivalent exact-count formulation; a single scalar cost is insufficient.
    All non-self product transitions advance topological key u+v.
    Resource exits never claim infeasibility or optimality.
    """
    start=time.perf_counter(); A=np.ascontiguousarray(A,float);B=np.ascontiguousarray(B,float)
    floor=max(np.linalg.norm(A[0]-B[0]),np.linalg.norm(A[-1]-B[-1]))
    if floor>d3+EPS:return dict(status='infeasible_endpoints',floor=float(floor))
    single=shortest_independent if kind=='CPS-2F' else independent_discrete
    ia=single(A,d1);ib=single(B,d2); lower=max(len(ia),len(ib))
    if certificate and discrete_frechet(A[ia],B[ib])<=d3+EPS:
        return finish(A,B,ia,ib,d1,d2,d3,kind,'optimal_independent_certificate',start,
                      lower_bound=lower,states=0,transitions=0)
    if certificate and kind=='CPS-2F':
        alternate,tests=alternate_certificate(A,B,d1,d2,d3,ia,ib)
        if alternate is not None:
            return finish(A,B,*alternate,d1,d2,d3,kind,'optimal_alternate_certificate',start,
                          lower_bound=lower,states=0,transitions=0,certificate_tests=tests)
    ga=chain_graph(A,d1,kind,max_chain_edges);gb=chain_graph(B,d2,kind,max_chain_edges)
    na,nb=len(A),len(B);end=(ga['size']-1,gb['size']-1)
    allowed=np.linalg.norm(A[:,None]-B[None,:],axis=2)<=d3+EPS
    values={(0,0):{0:0}}; parents={};heap=[(0,0,0)];queued={(0,0)};transitions=0
    while heap:
        _,u,v=heapq.heappop(heap); state=(u,v)
        if state==end:break
        current=values[state]
        for uu,ca in ga['out'][u]:
            for vv,cb in gb['out'][v]:
                if (uu==u and vv==v) or not allowed[uu%na,vv%nb]:continue
                transitions+=1
                if transitions>max_transitions:raise ResourceLimit('product transition budget exceeded')
                target=(uu,vv)
                if target not in values:
                    if len(values)>=max_states:raise ResourceLimit('reachable configuration budget exceeded')
                    values[target]={}
                dest=values[target]
                for r,z in current.items():
                    rr,zz=r+ca,z+cb
                    if zz<dest.get(rr,nb+1):dest[rr]=zz;parents[(uu,vv,rr)]=(u,v,r)
                if target not in queued:
                    heapq.heappush(heap,(uu+vv,uu,vv));queued.add(target)
        if time.perf_counter()-start>seconds:raise ResourceLimit('wall-time budget exceeded')
    if end not in values:return dict(status='infeasible_graph',floor=float(floor),states=len(values),transitions=transitions)
    r=min(values[end],key=lambda r:(max(r,values[end][r]),r+values[end][r],r))
    trace=[(*end,r)]
    while trace[-1]!=(0,0,0):trace.append(parents[trace[-1]])
    trace.reverse();ia=np.array(list(dict.fromkeys(u%na for u,v,r in trace)),int)
    ib=np.array(list(dict.fromkeys(v%nb for u,v,r in trace)),int)
    return finish(A,B,ia,ib,d1,d2,d3,kind,'optimal_configuration_graph',start,
                  lower_bound=lower,states=len(values),transitions=transitions,
                  auxiliary_A=ga['auxiliary'],auxiliary_B=gb['auxiliary'])

def finish(A,B,ia,ib,d1,d2,d3,kind,status,start,**stats):
    assert ia[0]==ib[0]==0 and ia[-1]==len(A)-1 and ib[-1]==len(B)-1
    assert np.all(np.diff(ia)>0) and np.all(np.diff(ib)>0)
    if kind=='CPS-2F':
        assert continuous_decision(A,A[ia],d1) and continuous_decision(B,B[ib],d2)
    else:assert discrete_frechet(A,A[ia])<=d1+EPS and discrete_frechet(B,B[ib])<=d2+EPS
    coupling,d=discrete_coupling(A[ia],B[ib]);assert d<=d3+EPS
    return dict(status=status,indices_A=ia,indices_B=ib,coupling=coupling,k=max(len(ia),len(ib)),
                dDF=d,verified=True,seconds=time.perf_counter()-start,**stats)


## 5. Validate before measuring

The tests enumerate all anchored subsequences on independent small random 3D curves and compare both joint optima against their respective discrete/continuous distance predicates. They force the graph path, bypassing the independent certificate. A collinear zero-tolerance example requires three vertices under discrete fidelity and only two under continuous fidelity. Compilation in this cell is excluded from experiment timings.

In [ ]:
import verify_cps_papers as verification
verification.solve_cps = solve_cps
verification.shortest_independent = shortest_independent
verification.independent_discrete = independent_discrete
verification.continuous_decision = continuous_decision
verification.discrete_frechet = discrete_frechet
checks = verification.run_checks()
display(checks)

### Exercise both configuration graphs on a protein display instance

This four-vertex window is drawn from the already decimated first pair. Disable both certificate shortcuts to exercise the actual graphs, including the thesis auxiliary points. This is a solver illustration, not an additional full-pair result; no display-instance number enters the comparison CSV.

In [ ]:
window_A = example['A'][:4].copy()
window_B = example['B_xyz'][:4].copy()
window_delta = float(.5*(np.linalg.norm(np.diff(window_A,axis=0),axis=1).mean()
                         + np.linalg.norm(np.diff(window_B,axis=0),axis=1).mean()))
window_delta3 = float(np.ceil(discrete_frechet(window_A, window_B)))
demonstrations = {}
display(pd.DataFrame([dict(pair=example['pair'], window='first four supplied vertices',
                          delta1=window_delta, delta2=window_delta, delta3=window_delta3)]))
for method in ('CPS-3F','CPS-2F'):
    demonstrations[method] = solve_cps(window_A, window_B, window_delta, window_delta,
                                       window_delta3, method, certificate=False)
display(pd.DataFrame({method: {k:v for k,v in result.items() if not isinstance(v,np.ndarray)}
                      for method,result in demonstrations.items()}).T)

## 6. Run all selected pairs

The following cell recomputes results; it does not load a prewritten result table. All requested method rows survive even when a configured resource budget is reached. Certificates and configuration-DAG executions have distinct status labels.

In [ ]:
import cps_notebook_run as experiment
# Run the solver definitions above; edits to those cells affect this experiment.
experiment.shortest_independent = shortest_independent
experiment.solve_cps = solve_cps
experiment.ResourceLimit = ResourceLimit
experiment.continuous_distance = continuous_distance
experiment.discrete_frechet = discrete_frechet
experiment.discrete_coupling = discrete_coupling
run_comparison = experiment.run_comparison
results, paths = run_comparison(data, OUT, ALPHAS, BUDGETS)
display(results.groupby(['method','status']).size().rename('rows').to_frame())
assert len(results) == len(instances) * len(ALPHAS) * 3
assert results[results.method != 'GCS'].query("status.str.startswith('optimal')", engine='python').pair_bound_verified.all()

### 6.1 Compression, auxiliary points and structural topology

**Sizes:** `nA=|A|`, `nB=|B|`, `kA=|A′|`, `kB=|B′|`. Compression factor is input count / output count; percentage removed is `100(1 − output/input)`. The joint objective remains `k=max(kA,kB)`; total compression is a separate diagnostic.

**Auxiliary points:** count the unique additional curve-parameter positions in the thesis extended curves, separately for A and B. Original vertices are excluded. `|A*|=|A|+auxiliary_A` and similarly for B. These points are not vertices of A′ or B′. Counts are computed for every CPS-2F case, including certificate cases, as diagnostic preprocessing outside solver timing. `auxiliary_graph_used` indicates a completed graph solution; resource-limited attempts do not have a completed graph solution. CPS-3F adds no auxiliary points.

**Structural topology proxy:** compare nonlocal Cα contact maps before and after simplification. A contact is a distance ≤8 Å between input vertices whose polymer sequence indices differ by at least 5. An [example of an 8 Å Cα contact definition](https://pmc.ncbi.nlm.nih.gov/articles/PMC6622159/) motivates the cutoff; the sequence exclusion and evaluation correspondence here are explicit experiment choices. To compare on the same vertex grid, place each discarded vertex along its shortcut using its original subcurve arc-length fraction. This geometric interpolation is used only for scoring; it does not reconstruct missing atoms or change solver input/output.

Report contact precision, recall and F1 for each chain, and a pooled F1 from summed true positives, false positives and false negatives. Undefined denominators remain missing, not perfect scores. Contact counts are exported to show sparse evidence at coarse strides. This measures preservation of nonlocal spatial relationships; **it does not certify knot type, entanglement, or absence of self-intersections**. Lower continuous Fréchet error and greater retained arc length give complementary geometric fidelity measures. Compare both methods at the exact same pair and δ1/δ2/δ3; no general topology advantage is assumed.

In [ ]:
from cps_quality import enrich_results, paired_comparison
CONTACT_CUTOFF = 8.0  # Å
CONTACT_SEQUENCE_SEPARATION = 5
results = enrich_results(results, data, paths, OUT,
                         cutoff=CONTACT_CUTOFF, separation=CONTACT_SEQUENCE_SEPARATION)
paired = paired_comparison(results, OUT)
auxiliary = results[results.method == 'CPS-2F'][[
    'pair','w','alpha','delta1','delta2','delta3','nA','nB','kA','kB',
    'auxiliary_A','auxiliary_B','auxiliary_total','extended_A','extended_B',
    'auxiliary_graph_used','status']]
auxiliary.to_csv(OUT/'auxiliary_counts.csv',index=False)
display(HTML(auxiliary.rename(columns={'nA':'|A|','nB':'|B|','kA':'|A′|','kB':'|B′|',
    'extended_A':'|A*|','extended_B':'|B*|'}).to_html(index=False,float_format=lambda x:f'{x:.3f}')))

### 6.2 Per-pair parameter table — sizes, budgets and auxiliary points

This is the main reference table for **every tested protein pair**: |A|, |B|, δ1, δ2, δ3, and the additional CPS-2F auxiliary points on A and B, including their total. The extended-curve sizes |A*| and |B*| are included. Separate blocks identify stride w and α; auxiliary counts depend on the actual fidelity thresholds, so they are not pooled across α values.

**Theoretical upper bounds** are listed beside the measured counts: `U_A = 2|A|(|A|−1)` and `U_B = 2|B|(|B|−1)`, from [Galit Gozoltzani, Chapter 3, Observation 4](../../docs/RESEARCH_SOURCES.md#thesis). Each vertex-centered sphere intersects each edge at most twice, assuming finite intersections. These are worst-case bounds on **added** points; they exclude original vertices and are not predictions. For example, 22 vertices give a bound of 924 added points, so the extended curve has at most 946 vertices. The measured count may be much smaller.

**Saved table:** computed from the same stride-8 inputs and thresholds used for this run.

In [ ]:
from cps_parameter_table import configured_parameter_table
pair_parameters, parameter_html = configured_parameter_table(data, ALPHAS, OUT)
display(HTML(parameter_html))
display(Markdown('[Open the complete parameter table](output/cps_papers_w8/pair_parameters.html) · '
                 '[Download all pair parameters and auxiliary counts (CSV)](output/cps_papers_w8/pair_parameters.csv)'))

## 7. Before/after results

The rendered table and CSV report both vertex counts, arc-length ratios, input/output discrete Fréchet distances, actual coupling links, fidelity distances, count overhead over global GCS and elapsed time. Continuous distances are decision brackets with width ≤ 10⁻⁵ Å; output tables display the upper bounds. Joint bound satisfaction verifies the solver. The empirical comparison is how vertex counts and achieved distances differ, including any independent pairing violations.

In [ ]:
from cps_notebook_plots import render_table, summary_plots, spatial_plots
display(HTML(render_table(results, OUT/'comparison.html')))
display(Markdown('[Full CSV](output/cps_papers_w8/comparison.csv) · '
 '[Standalone rendered table](output/cps_papers_w8/comparison.html) · '
 '[Input/residue map](output/cps_papers_w8/residues.csv) · '
 '[Selected vertices and discrete couplings](output/cps_papers_w8/paths.json)'))

### CPS-2F versus CPS-3F: matched comparisons

Each row identifies both proteins, sampling level, all three thresholds, input sizes, and both methods' output sizes. Positive `k_saved_by_2F` means CPS-2F uses a smaller maximum chain count. Positive `extra_removal_pp_2F` means more total vertices removed (percentage points); positive `contact_F1_gain_2F` means better contact preservation. Negative `continuous_error_change_2F` means lower geometric error. These can favor different methods. Missing results are not counted as losses or ties.

In [ ]:
paired_columns = ['pair','w','alpha','delta1','delta2','delta3','nA','nB',
    'kA_2F','kB_2F','kA_3F','kB_3F','k_saved_by_2F','extra_removal_pp_2F',
    'contact_F1_pooled_2F','contact_F1_pooled_3F','contact_F1_gain_2F',
    'continuous_error_change_2F','status_2F','status_3F']
display(HTML(paired[paired_columns].rename(columns={'nA':'|A|','nB':'|B|',
    'kA_2F':'|A′| CPS-2F','kB_2F':'|B′| CPS-2F',
    'kA_3F':'|A′| CPS-3F','kB_3F':'|B′| CPS-3F'}).to_html(
        index=False,na_rep='—',float_format=lambda x:f'{x:.3f}')))
display(Markdown('[Matched comparisons CSV](output/cps_papers_w8/cps2f_vs_cps3f.csv) · '
                 '[Auxiliary point counts](output/cps_papers_w8/auxiliary_counts.csv)'))

In [ ]:
valid = results[results.status.str.startswith('optimal')]
gcs = valid[valid.method == 'GCS']
display(pd.DataFrame({'quantity': ['independent pairs exceeding δ₃', 'successful joint verifications',
 'resource-limited rows', 'infeasible rows'], 'value': [int((~gcs.pair_bound_verified.astype(bool)).sum()),
 int(valid[valid.method!='GCS'].pair_bound_verified.sum()),
 int((results.status=='resource_limit').sum()), int(results.status.str.startswith('infeasible').sum())]}))
sampling = results.drop_duplicates(['pair','w'])[['B','w','role','nA','nB','edge_mean_before','edge_mean','edge_growth','edge_min','edge_max']]
display(sampling)
sampling.to_csv(OUT/'sampling_geometry.csv',index=False)

## 8. Figures: compression quality and structural fidelity

Each row compares CPS-2F with CPS-3F for one named pair. Every tick gives δ1, δ2 and δ3 in Å. Left: total vertex removal (higher is more compressed). Middle: pooled nonlocal contact F1 (higher preserves more contacts). Right: maximum continuous fidelity error across A and B (lower follows the input curves more closely). Coarse sampling can leave very few input contacts, so inspect the exported contact counts before drawing biological conclusions.

### 8.1 Boxplots: coupling budget and vertex cost

Panel (a) shows `d_dF(A′,B′)/δ3` for **all three methods**, including independent GCS. The dashed line at 1 is the pairing budget; values above it violate that budget. Panel (b) shows the requested boxplots of `k=max(|A′|,|B′|)` for each method, with independent GCS as the baseline. A companion boxplot makes the paired extra cost explicit: `k_method − k_GCS` for the **same input pair and tolerances**.

Only cases solved by all three methods enter these plots. An unsolved case is excluded from every method, so the distributions use identical inputs. The cohort table below identifies every protein pair, stride, α and δ1/δ2/δ3, including exclusions. Each point is a parameter case; repeated thresholds for the same protein pair are not independent biological observations. Boxes show medians and quartiles, whiskers extend to the furthest observation within 1.5 IQR, and all observations are overlaid.

In [ ]:
from cps_quality_plots import budget_vertex_boxplots
boxplot_cohort = budget_vertex_boxplots(results, OUT/'figures')
display(HTML(boxplot_cohort.to_html(index=False,float_format=lambda x:f'{x:.3f}')))
display(Markdown('[Exact pairs and thresholds](output/cps_papers_w8/boxplot_cohort.csv)'))

### 8.2 Publication figure: side-by-side grouped bars

For each α and stride, the left panel shows the achieved discrete coupling divided by δ3, with a dashed constraint line at 1. The right panel compares `k=max(|A′|,|B′|)` for Independent, CPS-2F and CPS-3F. Both panels use the same full protein-pair labels, rotated 90°. Colors and hatching agree across panels. Missing solver results are marked **NA**, not zero. The requested titles are descriptive headings; the plotted vertex costs show the actual overhead, including cases where CPS-3F costs more.

Each figure is exported as vector PDF and SVG, plus 300 dpi PNG. Mathematical labels use STIX math rendering and embedded PDF fonts, without requiring a local LaTeX installation. Exact δ1/δ2/δ3 are listed by figure and pair below.

In [ ]:
from cps_quality_plots import academic_dual_bars
dual_bar_parameters = academic_dual_bars(results, OUT/'figures')
display(HTML(dual_bar_parameters.to_html(index=False,float_format=lambda x:f'{x:.3f}')))
display(Markdown('[Per-pair thresholds](output/cps_papers_w8/dual_bar_parameters.csv) · '
                 '[α=1 vector PDF](output/cps_papers_w8/figures/dual_bars_w8_alpha1.pdf) · '
                 '[α=1 PNG](output/cps_papers_w8/figures/dual_bars_w8_alpha1.png)'))

In [ ]:
from cps_quality_plots import quality_plots
quality_plots(results, OUT/'figures')

### 8.3 Configuration graph G: measured size versus theory

Here graph size means **the number of vertices (valid configurations), $|V(G)|$**, before anchored-endpoint reachability pruning. All pairs are counted, including certificate and resource-limited cases. No joint optimization is rerun.

Write $m=|A|$, $n=|B|$, $m^*=|A^*|$, and $n^*=|B^*|$. CPS-3F has **two bars** (valid vertices and one bound); CPS-2F has **three bars**. Each panel has its own legend with the calculation:

- **Exact valid vertices:** $\sum_{p,q:\|a_p-b_q\|\leq\delta_3} c_A(p)c_B(q)$, where $c_A(p)$ counts owner positions of $A^*$ within $\delta_1$ of $a_p$, and similarly for $B$. This factorization counts the graph without materializing its Cartesian product. Counts use the solver's numerical tolerance.
- **Cartesian upper bound using measured extended sizes (CPS-2F only):** $m^*m n^*n$.
- **Worst-case theoretical upper bound:** CPS-2F uses $[m+2m(m-1)]m[n+2n(n-1)]n=O(m^3n^3)$; CPS-3F uses $m^2n^2$. CPS-3F has no auxiliary points and uses original vertices as owner positions, so its coincident bounds are shown by a single bar. These are bounds, not predictions of equality.

Source: [Galit Gozoltzani thesis, §5.1](../../docs/RESEARCH_SOURCES.md#thesis), together with the four-index discrete configuration graph in [Fan et al., Algorithm 1](../../docs/RESEARCH_SOURCES.md#chain-pair-simplification). The log scale makes the large gap to the worst-case bound visible. Independent GCS uses a different graph and is omitted from this joint-graph comparison.

The table also records **discovered states** and **examined transitions** from the existing solver run. These measure solver work, not full graph size: certificate runs build no joint graph, and missing counters were not recorded for interrupted runs. Full $|E(G)|$ is not measured; the thesis gives the asymptotic edge bound $O(|V|^2)$. Auxiliary counts exclude original vertices. Every row specifies the pair, stride, and all three distance budgets in ångströms.


In [ ]:
from cps_graph_sizes import graph_size_table, graph_size_plots, check_graph_counts, auxiliary_utilization_plots
graph_count_checks = check_graph_counts()
graph_sizes = graph_size_table(data, results, OUT)
print(f'Graph-count verification: {graph_count_checks} brute-force checks passed.')
graph_size_plots(graph_sizes, OUT/'figures')
display(Markdown('#### Auxiliary-point utilization percentage\n\n'
    'For each CPS-2F pair, the bars show **100 × added auxiliary points / theoretical auxiliary bound**, '
    'separately for A and B. The bounds are 2|A|(|A|−1) and 2|B|(|B|−1); original vertices are excluded. '
    'CPS-3F has no auxiliary points and is omitted. Zero denominators are reported as N/A.'))
auxiliary_utilization = auxiliary_utilization_plots(graph_sizes, OUT/'figures')
display(HTML(auxiliary_utilization.to_html(index=False, float_format=lambda x:f'{x:.3f}')))
display(Markdown(f'[Auxiliary utilization and exact budgets]({OUT.relative_to(ROOT).as_posix()}/auxiliary_utilization.csv)'))
display(HTML(graph_sizes.to_html(index=False, float_format=lambda x:f'{x:.5g}')))
display(Markdown(f'[Graph counts and exact budgets]({OUT.relative_to(ROOT).as_posix()}/graph_sizes.csv)'))
graph_manifest = dict(ws=list(WS), alphas=list(ALPHAS), checks=graph_count_checks,
    rows=len(graph_sizes), definition='full valid vertices before reachability pruning',
    sha256={name:hashlib.sha256((ROOT/name).read_bytes()).hexdigest()
            for name in ['src/cps_graph_sizes.py','src/cps_paper_algorithms.py','src/curve_algorithms.py']},
    comparison_sha256=hashlib.sha256((OUT/'comparison.csv').read_bytes()).hexdigest())
(OUT/'graph_size_manifest.json').write_text(json.dumps(graph_manifest,indent=2))


## 9. 3D before/after figures for every pair

At α=1, each pair has an input panel and one panel per algorithm. The four panels share the same viewpoint, x/y/z limits, and equal axis scales. Dotted links show the discrete Fréchet coupling, including repeated indices. Faint lines show the full prepared decimated input. Numerical measurements use 3D coordinates, never a projection.

In [ ]:
for w in WS:
    spatial_plots(data, paths, OUT/'figures', alpha=ALPHAS[0], w=w)

## 10. Interpretation, limitations and planned comparisons

- Compare global GCS with CPS-2F to isolate the price of a joint constraint under the same continuous fidelity. Compare CPS-2F with CPS-3F to quantify the effect of replacing discrete fidelity by continuous fidelity. Do not interpret different-sized outputs as an unusable coupling: the papers use discrete Fréchet traversal, not an equal-size zip.
- Report what the measurements show. Independent outputs need not violate δ₃ in every run; a slack threshold and coarse input can make both methods coincide. Joint solvers enforce a threshold but do not minimize their achieved pairing distance.
- `w=8` is an initial executable coarse-trace comparison. `w=1` is available as the undecimated control; `w=2,4` are the next biologically relevant comparisons. No w≤4 headline is inferred from this default run. Explicit graph budgets may limit larger instances; raise them deliberately or implement the source's faster/sparser algorithms without changing the optimization problem.
- The mean chord grows sublinearly with stride because the backbone is folded. `edge_growth`, `edge_min` and `edge_max` make this visible. The appended final vertex creates a short stub when the stride misses the endpoint. At mean edges above roughly 10 Å, regard the curve as a coarse trace that loses helical detail.
- The default retains all supplied vertices and treats gap-crossing edges as straight segments. The optional qualified mode removes regions outside common continuous intervals. Neither mode reconstructs missing residues. Table 3 supplements retain unequal full-chain lengths, but do not recover the unavailable historical extraction.
- The supplied pairs use their provided Kabsch frame; Table 3 supplements use cached US-align transforms fitted on qualified intervals and applied to full curves. These pair-specific frames, present-day coordinates and decimation mean that historical distances and counts need not match.
- The local thesis is an annotated draft. Graph optimality and global continuous feasibility are separately tested; the asserted equivalence of its auxiliary-point graph to the full continuous problem rests on the thesis's discretization claim. Remaining source annotations should be resolved before treating it as an independently established theorem.
- Endpoints are anchored here for all three methods. Fan and the thesis discuss unanchored extensions; results for those variants should be separate. Historical Table 3 also uses unequal δ₁ and δ₂, whereas this bond-scaled comparison intentionally sets them equal.

Reproducibility: configuration, file hashes and all validation counts are recorded below. Supporting scripts are local; no network request is needed to rerun against the cached audited inputs.

In [ ]:
source_files = [ROOT/'src/cps_paper_algorithms.py', ROOT/'src/curve_algorithms.py',
 ROOT/'src/cps_notebook_data.py', ROOT/'src/cps_notebook_run.py', ROOT/'src/cps_quality.py',
 ROOT/'src/cps_quality_plots.py', ROOT/'src/cps_notebook_plots.py',
 ROOT/'data/decimated protein backbones/decimated_curves.csv',
 ROOT/'papers/M_Sc__Thesis___Galit_Gozoltzani.pdf', ROOT/'papers/global curve simplification.pdf',
 ROOT/'papers/chain_pair_simplification.pdf']
manifest = dict(ws=list(WS), alphas=list(ALPHAS), include_table3=INCLUDE_TABLE3,
 input_scope=INPUT_SCOPE, units='angstrom', endpoints='anchored to prepared inputs', budgets=BUDGETS,
 rows=len(results), pairs=results.pair.nunique(), checks=checks,
 contact_cutoff=CONTACT_CUTOFF, contact_sequence_separation=CONTACT_SEQUENCE_SEPARATION,
 sha256={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files})
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
display(manifest)